In [ ]:
!pip install -q transformers accelerate mlflow datasets

from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/FakeNewsProject'

import torch, pandas as pd, numpy as np
print(f"GPU : {torch.cuda.get_device_name(0)}")

train = pd.read_csv(f'{BASE}/data/processed/train.csv')
val   = pd.read_csv(f'{BASE}/data/processed/val.csv')
test  = pd.read_csv(f'{BASE}/data/processed/test.csv')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 101.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 116.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.5/838.5 kB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

dataset Pytorch pour BERT

In [ ]:
from torch.utils.data import Dataset
from transformers import AutoTokenizer

MODEL_NAME = 'bert-base-uncased'  # changer en 'roberta-base' pour RoBERTa
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

class NewsDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=512):
        # Use 'clean_text' column as it appears in the provided dataframes
        texts          = df['clean_text'].fillna('').tolist()
        self.encodings = tokenizer(texts, truncation=True, padding=True,
                                    max_length=max_len, return_tensors='pt')
        self.labels    = torch.tensor(df['label'].values, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

train_ds = NewsDataset(train, tokenizer)
val_ds   = NewsDataset(val,   tokenizer)
test_ds  = NewsDataset(test,  tokenizer)
print(f"✓ Datasets créés — Train: {len(train_ds)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✓ Datasets créés — Train: 26938


In [ ]:
!pip uninstall transformers accelerate -y
!pip install transformers==4.41.0 accelerate==0.30.0 -q

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.4/302.4 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 66.0 MB/s eta 0:00:00


In [ ]:
import transformers, accelerate
print("transformers :", transformers.__version__)  # doit afficher 4.41.0
print("accelerate   :", accelerate.__version__)    # doit afficher 0.30.0

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

transformers : 4.41.0
accelerate   : 0.30.0


FineTunning avec HugginFace

In [ ]:
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup, AutoTokenizer
from torch.utils.data import DataLoader
from torch.optim import AdamW
from sklearn.metrics import f1_score, roc_auc_score
import numpy as np, torch, os, json

# Add BASE definition here for robustness if kernel state is lost
BASE = '/content/drive/MyDrive/FakeNewsProject'

# Define MODEL_NAME here for robustness if kernel state is lost
MODEL_NAME = 'bert-base-uncased'

# ── Configuration ──────────────────────────────────────────────
EPOCHS    = 3
BATCH_TR  = 16
BATCH_EV  = 32
LR        = 2e-5
DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_PATH = f'{BASE}/models/bert_final'
CKPT_PATH = f'{BASE}/models/checkpoint_state.json'
os.makedirs(SAVE_PATH, exist_ok=True)
os.makedirs(f'{BASE}/models', exist_ok=True)
print(f"Device : {DEVICE}")

# ── Tokenizer ─────────────────────────────────────────────────
tok_bert = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"✓ Tokenizer chargé")

# ── Charger état de reprise si existant ───────────────────────
def load_state():
    if os.path.exists(CKPT_PATH):
        with open(CKPT_PATH) as f:
            state = json.load(f)
        print(f"✓ Reprise depuis epoch {state['last_epoch']+1}, meilleur F1={state['best_f1']:.4f}")
        return state
    return {"last_epoch": -1, "best_f1": 0.0}

def save_state(epoch, best_f1):
    with open(CKPT_PATH, 'w') as f:
        json.dump({"last_epoch": epoch, "best_f1": best_f1}, f)

state   = load_state()
START   = state["last_epoch"] + 1
best_f1 = state["best_f1"]

# ── Charger modèle ────────────────────────────────────────────
if START > 0:
    print(f"Chargement checkpoint epoch {START}...")
    model = AutoModelForSequenceClassification.from_pretrained(SAVE_PATH).to(DEVICE)
else:
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2).to(DEVICE)

if START >= EPOCHS:
    print("✓ Entraînement déjà terminé !")
else:
    # ── DataLoaders ───────────────────────────────────────────
    train_loader = DataLoader(train_ds, batch_size=BATCH_TR, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_EV, shuffle=False)

    # ── Optimizer + Scheduler ────────────────────────────────
    optimizer    = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total_steps  = len(train_loader) * EPOCHS
    warmup_steps = int(0.1 * total_steps)
    scheduler    = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps   = warmup_steps,
        num_training_steps = total_steps
    )
    scaler = torch.amp.GradScaler('cuda')

    # ── Fonction évaluation ───────────────────────────────────
    def evaluate(model, loader):
        model.eval()
        all_preds, all_labels, all_probas = [], [], []
        total_loss = 0
        with torch.no_grad():
            for batch in loader:
                ids    = batch['input_ids'].to(DEVICE)
                mask   = batch['attention_mask'].to(DEVICE)
                labels = batch['labels'].to(DEVICE)
                out    = model(input_ids=ids, attention_mask=mask, labels=labels)
                total_loss += out.loss.item()
                probs  = torch.softmax(out.logits, dim=-1)[:,1].cpu().numpy()
                preds  = np.argmax(out.logits.cpu().numpy(), axis=-1)
                all_preds.extend(preds)
                all_labels.extend(labels.cpu().numpy())
                all_probas.extend(probs)
        return (total_loss / len(loader),
                f1_score(all_labels, all_preds, average='macro'),
                roc_auc_score(all_labels, all_probas))

    # ── Boucle entraînement ───────────────────────────────────
    print("="*58)
    print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Val Loss':>8} | {'F1':>6} | {'AUC':>6}")
    print("="*58)

    for epoch in range(START, EPOCHS):
        model.train()
        total_train_loss = 0

        for step, batch in enumerate(train_loader):
            ids    = batch['input_ids'].to(DEVICE)
            mask   = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)

            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                out  = model(input_ids=ids, attention_mask=mask, labels=labels)
                loss = out.loss

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total_train_loss += loss.item()

            if step % 200 == 0:
                print(f"  Epoch {epoch+1} — step {step}/{len(train_loader)} — loss: {loss.item():.4f}")

        # ── Évaluation fin d'epoch ────────────────────────────
        val_loss, val_f1, val_auc = evaluate(model, val_loader)
        train_loss = total_train_loss / len(train_loader)
        print(f"{epoch+1:>6} | {train_loss:>10.4f} | {val_loss:>8.4f} | {val_f1:>6.4f} | {val_auc:>6.4f}")

        # ── Sauvegarde après chaque epoch ─────────────────────
        model.save_pretrained(SAVE_PATH)
        tok_bert.save_pretrained(SAVE_PATH)
        save_state(epoch, val_f1 if val_f1 > best_f1 else best_f1)

        if val_f1 > best_f1:
            best_f1 = val_f1
            print(f"  ✓ Meilleur modèle sauvegardé (F1={best_f1:.4f})")

    print("="*58)
    print(f"✓ Entraînement terminé — Meilleur F1 : {best_f1:.4f}")
    print(f"✓ Modèle sauvegardé dans : {SAVE_PATH}")

Device : cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✓ Tokenizer chargé
✓ Reprise depuis epoch 3, meilleur F1=0.9979
Chargement checkpoint epoch 3...
✓ Entraînement déjà terminé !


In [ ]:
from torch.utils.data import Dataset
from transformers import AutoTokenizer
import torch
import pandas as pd

# Ensure BASE is defined for robustness if kernel state is lost
BASE = '/content/drive/MyDrive/FakeNewsProject'

# Ensure MODEL_NAME is defined (it's also defined in aImtjbmYUySU)
MODEL_NAME = 'bert-base-uncased'

# Ensure tokenizer is available (it's also defined as tok_bert in aImtjbmYUySU)
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

# Ensure dataframes are loaded for robustness if kernel state is lost
train = pd.read_csv(f'{BASE}/data/processed/train.csv')
val   = pd.read_csv(f'{BASE}/data/processed/val.csv')
test  = pd.read_csv(f'{BASE}/data/processed/test.csv')

class NewsDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=512):
        texts          = df['clean_text'].fillna('').tolist()
        self.encodings = tokenizer(texts, truncation=True, padding=True,
                                    max_length=max_len, return_tensors='pt')
        self.labels    = torch.tensor(df['label'].values, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

train_ds = NewsDataset(train, tokenizer)
val_ds   = NewsDataset(val,   tokenizer)
test_ds  = NewsDataset(test,  tokenizer)
print(f"✓ Datasets créés — Train: {len(train_ds)}")

✓ Datasets créés — Train: 26938


In [ ]:
import os
import torch
import numpy as np
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, roc_auc_score, f1_score
import mlflow

BASE = '/content/drive/MyDrive/FakeNewsProject'
os.makedirs(f'{BASE}/models/bert_final', exist_ok=True)

# 1. Créer le DataLoader pour le jeu de test
test_loader = DataLoader(test_ds, batch_size=BATCH_EV, shuffle=False)

print("Génération des prédictions sur le jeu de test...")
model.eval()
all_preds, all_labels, all_probas = [], [], []

# 2. Boucle de prédiction (remplace trainer.predict)
with torch.no_grad():
    for batch in test_loader:
        ids    = batch['input_ids'].to(DEVICE)
        mask   = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)

        out = model(input_ids=ids, attention_mask=mask)
        logits = out.logits

        # Calculer les probabilités et les classes prédites
        probs = torch.softmax(logits, dim=-1)[:,1].cpu().numpy()
        preds = np.argmax(logits.cpu().numpy(), axis=-1)

        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
        all_probas.extend(probs)

# Conversion en tableaux NumPy pour les métriques
labels = np.array(all_labels)
preds  = np.array(all_preds)
probas = np.array(all_probas)

# 3. Évaluation
print("\n=== BERT TEST RESULTS ===")
print(classification_report(labels, preds, target_names=['FAKE','REAL']))
print(f"AUC-ROC : {roc_auc_score(labels, probas):.4f}")

# 4. Sauvegarder le modèle et tokenizer (sans utiliser le trainer)
model.save_pretrained(f'{BASE}/models/bert_final')
tok_bert.save_pretrained(f'{BASE}/models/bert_final') # Utilisation de 'tok_bert' que tu as défini plus haut

# 5. MLflow
mlflow.set_tracking_uri(f'{BASE}/mlruns')
mlflow.set_experiment("fake-news-detection")
with mlflow.start_run(run_name="BERT_base_uncased"):
    mlflow.log_params({"model": MODEL_NAME, "epochs": EPOCHS, "lr": LR, "batch": BATCH_TR})
    mlflow.log_metrics({
        "f1_macro": f1_score(labels, preds, average='macro'),
        "f1_fake" : f1_score(labels, preds, pos_label=0),
        "auc_roc" : roc_auc_score(labels, probas)
    })

print(f"\n✓ BERT sauvegardé avec succès dans {BASE}/models/bert_final")

Génération des prédictions sur le jeu de test...

=== BERT TEST RESULTS ===
              precision    recall  f1-score   support

        FAKE       1.00      0.99      1.00      4696
        REAL       0.99      1.00      1.00      4284

    accuracy                           1.00      8980
   macro avg       1.00      1.00      1.00      8980
weighted avg       1.00      1.00      1.00      8980

AUC-ROC : 0.9998


/usr/local/lib/python3.12/dist-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)



✓ BERT sauvegardé avec succès dans /content/drive/MyDrive/FakeNewsProject/models/bert_final
